# Telcovantage — GPU TrOCR Server

Runs TrOCR models on Colab GPU and exposes them via ngrok.

**Workflow:**
1. Run all cells below
2. Copy the `REMOTE_TROCR_URL` printed at the end
3. On your laptop: `set REMOTE_TROCR_URL=<url>` then `py server.py`

---
## Cell 1: Install dependencies

In [ ]:
!pip install -q fastapi uvicorn pyngrok

---
## Cell 2: Clone your repo

In [ ]:
import os
REPO_URL = "https://github.com/jonrenzo/Telcovantage-Site-Map-Reader"
REPO_DIR = "/content/Telcovantage-Site-Map-Reader"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Already cloned at {REPO_DIR}")
    !cd {REPO_DIR} && git pull

---
## Cell 3: Install project Python dependencies

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt
print("Dependencies installed.")

---
## Cell 4: Set ngrok auth token

1. Go to https://dashboard.ngrok.com → get your **authtoken**
2. Paste it below between the quotes

In [ ]:
NGROK_AUTH_TOKEN = ""  # ← PASTE YOUR NGROK TOKEN HERE

from pyngrok import ngrok
ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok auth set.")

---
## Cell 5: Start TrOCR server + ngrok tunnel (non-blocking)

In [ ]:
import threading, time, sys, requests
sys.path.insert(0, REPO_DIR)

import uvicorn
from colab.server import app

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

t = threading.Thread(target=run_server, daemon=True)
t.start()

# Wait for server to start
for i in range(30):
    time.sleep(1)
    try:
        r = requests.get("http://localhost:8000/health", timeout=3)
        if r.status_code == 200:
            print("Server started on port 8000.")
            break
    except Exception:
        pass

# Pre-warm the model
print("Loading TrOCR model on GPU (first run may take ~15-30s)...")
resp = requests.post("http://localhost:8000/ocr/pole",
    json={"segments": [{"x1":0,"y1":0,"x2":10,"y2":0},
                      {"x1":5,"y1":0,"x2":5,"y2":20}],
          "bbox": [0,0,10,20], "auto_rotate": False}, timeout=180)
print("Model loaded:", resp.status_code)

from pyngrok import ngrok
ngrok.kill()
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print(f"\n{'='*60}")
print(f"  REMOTE_TROCR_URL = {public_url}")
print(f"{'='*60}")
print()
print("Set this on your laptop:")
print(f"  $env:REMOTE_TROCR_URL = \"{public_url}\"")
print(f"  py server.py")

---
## Cell 6: Keep-alive (non-blocking)

In [ ]:
from IPython.display import display, Javascript
display(Javascript("""
if (!window._colabKeepalive) {
  window._colabKeepalive = setInterval(function(){}, 60000);
  console.log('Keepalive started.');
}
"""))
print("Keepalive active (browser JS, does not block kernel).")